In [1]:
# set parameters
from glob import glob

params = {
    'csv_files': glob('../data/csvs/*.csv'), #csv files in raw image space to be transformed
    'mesh_path': '../code/ccf_files/CCF_meshes/json_verts_float', #location of ccf meshes for plotting
    "smartspim_template": '../data/lightsheet_template_ccf_registration/smartspim_lca_template_25.nii.gz', # location of smartspim template
    "ccf_template": '../data/lightsheet_template_ccf_registration/ccf_average_template_25.nii.gz', #location of ccf template
    'template_to_ccf': [
        '../data/lightsheet_template_ccf_registration/spim_template_to_ccf_syn_0GenericAffine.mat',
        '../data/lightsheet_template_ccf_registration/syn_1InverseWarp.nii.gz'
    ], #transforms for ls_tempalte -> ccf 
    'downsample': 3, #the level used for registering the raw image to ls_template
    'ccf_res': 25,
    'scaling': [16 / 25, 14.4 / 25, 14.4 / 25], #resolution change from downsampled image to ants object
    'institute': 'AIND' #institute that imaged brain
}

In [4]:
# loop over csv files and register
import os
import re
import numpy as np
from utils import utils


registered_cells = {}

for file in params['csv_files']:
    
    lt_id = re.findall('[0-9]{6}', file)[0]
    print(f'registering dataset: {lt_id}')
    
    acquisition_file = glob(f'../data/SmartSPIM_{lt_id}*/acquisition.json')[0]
    zarr_path = glob(f'../data/SmartSPIM_{lt_id}*/image_tile_fusing/OMEZarr/*.zarr')[0]
    
    
    print(f'location of acquisition file: {acquisition_file}\nlocation of zarr: {zarr_path}')
    
    ls_to_template = [
        glob(f'../data/SmartSPIM_{lt_id}*/image_atlas_alignment/**/ls_to_template_SyN_0GenericAffine.mat')[0],
        glob(f'../data/SmartSPIM_{lt_id}*/image_atlas_alignment/**/ls_to_template_SyN_1InverseWarp.nii.gz')[0]
    ]
    
    print(f'lightsheet to template transforms: {ls_to_template}\n\n')
    
    res_metadata = f"{zarr_path}/0/.zarray"
    
    input_res = utils.read_json_as_dict(res_metadata)["shape"]

    # input res is returned in order tczyx, here we use xzy
    # where z is the imaging axis
    res = [
        input_res[-3],
        input_res[-2],
        input_res[-1],
    ]

    # get dimensions at registered scale
    ds = 2**params['downsample']
    reg_dims = [dim / ds for dim in res]
    
    # get orientation information
    orientation = utils.read_json_as_dict(acquisition_file)
    orient = utils.get_orientation(orientation['axes'])
    template_params = utils.get_template_info(params["smartspim_template"])
    
    _, swapped, mat = utils.get_orientation_transform(
        orient, 
        template_params["orientation"]
    )
    
    raw_cells = utils.read_cells_from_csv(
        cell_likelihoods_path=file,
        reg_dims=reg_dims,
        ds=ds,
        orient=orient,
        orient_matrix=mat,
        institute=params['institute'],
    )
    
    scaled_cells = utils.scale_cells(raw_cells, params['scaling'])
    scaled_cells = np.array(scaled_cells)
    orient_cells = scaled_cells[:, swapped]


    #Converting oriented cells into ANTs physical space
    template_params = utils.get_template_info(params["smartspim_template"])
    ants_cells = utils.convert_to_ants_space(template_params, orient_cells)
    
    #Registering Cells to SmartSPIM template
    template_cells = utils.apply_transforms_to_points(
        ants_cells, ls_to_template, invert=(True, False)
    )

    #Convert template cells into CCF space and orientation
    ccf_pts = utils.apply_transforms_to_points(
        template_cells, params['template_to_ccf'], invert=(True, False)
    )

    #Convert cells back into index space
    ccf_params = utils.get_template_info(params["ccf_template"])
    ccf_cells = utils.convert_from_ants_space(ccf_params, ccf_pts)

    _, swapped, _ = utils.get_orientation_transform(
        template_params["orientation"], ccf_params["orientation"]
    )
    
    cells_transformed = ccf_cells[:, swapped]
    
    registered_cells[lt_id] = cells_transformed
    

registering dataset: 755072
location of acquisition file: ../data/SmartSPIM_755072_2024-11-13_21-29-44_stitched_2024-11-16_03-19-42/acquisition.json
location of zarr: ../data/SmartSPIM_755072_2024-11-13_21-29-44_stitched_2024-11-16_03-19-42/image_tile_fusing/OMEZarr/Ex_639_Em_667.zarr
lightsheet to template transforms: ['../data/SmartSPIM_755072_2024-11-13_21-29-44_stitched_2024-11-16_03-19-42/image_atlas_alignment/Ex_639_Em_667/ls_to_template_SyN_0GenericAffine.mat', '../data/SmartSPIM_755072_2024-11-13_21-29-44_stitched_2024-11-16_03-19-42/image_atlas_alignment/Ex_639_Em_667/ls_to_template_SyN_1InverseWarp.nii.gz']
registering dataset: 751017
location of acquisition file: ../data/SmartSPIM_751017_2024-11-14_19-17-29_stitched_2024-11-16_05-13-56/acquisition.json
location of zarr: ../data/SmartSPIM_751017_2024-11-14_19-17-29_stitched_2024-11-16_05-13-56/image_tile_fusing/OMEZarr/Ex_561_Em_593.zarr
lightsheet to template transforms: ['../data/SmartSPIM_751017_2024-11-14_19-17-29_stitche

In [6]:
# Functions for loading mesh for viewing
import os
import json
import pickle

def get_region_lists():
    """
    Import list of acronyms of brain regions
    """
    
    CCF_dir = '../code/ccf_files/CCF_meshes'
    
    
    # Reading non-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "non_crossing_structures"), "rb") as f:
        hemi_struct = pickle.load(f)
        hemi_struct.remove(1051)  # don't know why this is being done
        hemi_labeled = [(s, "hemi") for s in hemi_struct]

    # Reading mid-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "mid_crossing_structures"), "rb") as f:
        u = pickle._Unpickler(f)
        u.encoding = "latin1"
        mid_struct = u.load()
        mid_labeled = [(s, "mid") for s in mid_struct]

    return hemi_labeled + mid_labeled

def load_json_mesh(root, struct):
    """
    Loads meshes that are stored in json files like the CCF meshes

    Parameters
    ----------
    root : str
        path to mesh
    struct: str
        the region id for mesh you want to load

    Returns
    -------
    tuple
        The vertices and faces of a given mesh

    """
        
    region_metadata = get_region_lists()
    region_dict = {i[0]: i[1] for i in region_metadata}
    fname = os.path.join(root, f"{struct}.json")
        
    with open(fname) as f:
        structure_data = json.loads(f.read())
        verts, faces = (
            np.array(structure_data[struct]["vertices"]),
            np.array(structure_data[struct]["faces"]),
        )
            
        if region_dict[int(struct)] == 'hemi':
            offset = faces.max() + 1
            faces = np.vstack((faces, faces + offset))
                
            verts_2 = verts.copy()
            verts_2[:, 0] = verts_2[:, 0] + (5700 - verts_2[:, 0]) * 2
            verts = np.vstack((verts, verts_2))
            
    return verts, faces

In [19]:
import k3d
import matplotlib as mpl


def rgb_to_hex(r,g,b):
    # Convert to a hexadecimal string
    hex_color = f'{r:02x}{g:02x}{b:02x}'
    # Convert the hexadecimal string to an integer in base-16
    color_int = int(hex_color, 16)
    return color_int


# Initialize plot
plot = k3d.plot()

## Plot meshes
verts, faces = load_json_mesh(params['mesh_path'], '997')
verts = verts[:, [2, 1, 0]] / params['ccf_res']

mesh = k3d.mesh(verts, faces, color=rgb_to_hex(128,128,128), opacity=0.05)
plot += mesh

verts, faces = load_json_mesh(params['mesh_path'], '771')
verts = verts[:, [2, 1, 0]] / params['ccf_res']

mesh = k3d.mesh(verts, faces, color=rgb_to_hex(0, 255, 255), opacity=0.2)
plot += mesh


# plot annotations
colors = mpl.colormaps['rainbow'].resampled(len(registered_cells))
cmap = np.array(colors(len(registered_cells))) * 255
cmap = [tuple(i[:3]) for i in cmap.astype(int)]
c = 0

for key, cells_transformed in registered_cells.items():
    region_pts = np.array(cells_transformed)
    pts = k3d.points(positions = region_pts.astype('float32'), point_size = 2, color = rgb_to_hex(*cmap[c]))
    plot += pts
    c += 1


plot.display()

Output()